
## Initalizing bronze layer location and Installing the required libraries

In [70]:
import requests
import json
from datetime import *
from pyspark.sql.functions import *
from notebookutils import mssparkutils

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 72, Finished, Available, Finished, False)

In [1]:
tables_str = tables_str
raw = raw
bronze = bronze
lakehouse = lakehouse
workspace = workspace

Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, 63ab1aaf-b1d8-4c65-9055-148326423eba, 3, Finished, Available, Finished, False)

In [72]:
BASE_URL = "https://hapi.fhir.org/baseR4"

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 74, Finished, Available, Finished, False)

In [73]:
today = datetime.now(UTC)
dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(3)]

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 75, Finished, Available, Finished, False)

# Raw Layer
### Loading the data to raw layer
1. Added additional metadata column extraction_timestamp and api_url_or_params.

In [74]:
def fetch_fhir_data(Table, last_updated):
    
    url = f"{BASE_URL}/{Table}?_lastUpdated=ge{last_updated}&_count=100"
    api_calls = []

    while url:
        api_calls.append(url)

        response = requests.get(url)

        if response.status_code != 200:
            print(f"API failed: {response.status_code}")
            break

        data = response.json()

        ts = datetime.utcnow().strftime("%H%M%S")


        raw_path = f"Files/{raw}/{Table.lower()}/{last_updated}/{ts}"

        df = spark.createDataFrame([{
            "raw_json": data,
            "load_date": date,
            "extraction_timestamp": datetime.utcnow(),
            "api_url": url
        }])

        df.write.mode("append").json(raw_path)

        # pagination
        next_link = None
        if "link" in data:
            for link in data["link"]:
                if link.get("relation") == "next":
                    next_link = link.get("url")

        if next_link == url:  # safety check
            break

        url = next_link


for Table in Tables:
    for date in dates:
        fetch_fhir_data(Table, date)

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 76, Finished, Available, Finished, False)

# Bronze layer

### Loading the data to bronze layer from raw

In [75]:

for Table in Tables:

    raw_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Files/{raw}/{Table.lower()}/*/*"

    df = spark.read.json(raw_path)

    bronze_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Files/{bronze}/{Table.lower()}"
                                                                                                    
    df.write.format("delta") \
        .mode("append") \
        .save(bronze_path)

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 77, Finished, Available, Finished, False)

In [76]:
# raw_path = f"abfss://Fabric_Assignment_Synapx@onelake.dfs.fabric.microsoft.com/FHIR_Lakehouse.Lakehouse/Files/raw/Patient/*/*/*.json"

# df = spark.read.json(raw_path)
# display(df)

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc509bca-9612-4516-b01c-e6a90395f2cc)

In [77]:
# bronze_path = "abfss://Fabric_Assignment_Synapx@onelake.dfs.fabric.microsoft.com/FHIR_Lakehouse.Lakehouse/Files/bronze/patient"

# df = spark.read.format("delta").load(bronze_path)

# display(df)

StatementMeta(, 0687c36c-8c60-49b0-9866-9d9fe4ca30ee, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 99311134-903b-4797-917a-f17cfecfbe7d)